# Faza I – Paraprocesimi i të dhënave (Eurostat)

**Dataset:** `estat_tour_occ_nim_en.csv`  
**Qëllimi:** analizimi i strukturës dhe cilësisë së të dhënave, pastrimi, trajtimi i vlerave që mungojnë, mostrimi, agregimi, krijimi/përzgjedhja e veçorive, diskretizimi, binarizimi, PCA dhe transformimi.

## Si të përdoret
1. Vendoseni këtë notebook në të njëjtën dosje me skedarin CSV.
2. Instaloni bibliotekat nëse nevojiten: `pip install pandas numpy matplotlib scikit-learn`.
3. Ekzekutoni qelizat nga lart-poshtë.
4. Rezultatet eksportohen në dosjen `output_phase1`.

## 1. Importimi i bibliotekave dhe ngarkimi i datasetit

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)

INPUT_FILE = Path("estat_tour_occ_nim_en.csv")
if not INPUT_FILE.exists():
    fallback = Path("estat_tour_occ_nim_en.csv")
    if fallback.exists():
        INPUT_FILE = fallback
    else:
        raise FileNotFoundError("Vendoseni CSV-në në të njëjtën dosje me notebook-un ose ndryshoni INPUT_FILE.")

OUTPUT_DIR = Path("output_phase1")
OUTPUT_DIR.mkdir(exist_ok=True)

df_original = pd.read_csv(INPUT_FILE, low_memory=False)
df = df_original.copy()

print("Forma:", df.shape)
display(df.head())
print(df.columns.tolist())

## 2. Tipet e të dhënave dhe fjalori i variablave

In [ ]:
df.info()

type_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
    "missing_values": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [(df[c].isna().mean() * 100).round(2) for c in df.columns],
})
display(type_report)
type_report.to_csv(OUTPUT_DIR / "01_data_types_report.csv", index=False)

column_descriptions = {
    "DATAFLOW": "Identifikuesi i rrjedhës së të dhënave",
    "LAST UPDATE": "Koha e përditësimit të datasetit",
    "freq": "Frekuenca (M = mujore)",
    "c_resid": "Kategoria e rezidentëve",
    "unit": "Njësia/treguesi statistikor",
    "nace_r2": "Kategoria e aktivitetit ekonomik",
    "geo": "Shteti/territori",
    "TIME_PERIOD": "Periudha kohore (vit-muaj)",
    "OBS_VALUE": "Vlera numerike e vëzhgimit",
    "OBS_FLAG": "Shenja/flag-u i vëzhgimit",
    "CONF_STATUS": "Statusi i konfidencialitetit",
}
variable_report = pd.DataFrame({
    "column": df.columns,
    "description": [column_descriptions.get(c, "Përshkrim për t'u verifikuar") for c in df.columns],
    "actual_dtype": [str(df[c].dtype) for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
})
display(variable_report)
variable_report.to_csv(OUTPUT_DIR / "02_variable_dictionary.csv", index=False)